# Seeds and Augmentation — putting error bars on the headline claim

**Two open questions, both currently unanswerable from the evidence on hand.**

1. **Is the sweep winner's advantage real?** The 12-run sweep found a configuration at
   PSNR 37.11 dB against V12's 32.95 dB — a +4.16 dB gap. Both numbers come from a *single*
   training run. V7 and V9 were the same configuration trained twice and Moment-2 moved from
   +18.4% to +2.5%, so single-run gaps in this project are known to be unreliable. Until both
   sides have a spread, the +4.16 dB cannot be distinguished from seed noise.

2. **Does augmentation help?** The pipeline had none until now, on 14 cubes / 1050 training
   items, with holdout M2 std of 14.3% and a hallucination artifact attributed to small data.
   `FITSChannelDataset(augment=True)` applies the 8-element dihedral group (4 rotations x
   optional flip) identically to dirty and clean — lossless, since arbitrary-angle rotation
   would need interpolation and would smooth the very noise the model must learn to remove.

**Design.** Three configurations x 3 seeds = 9 training runs. Only the training seed varies;
the cube split is held fixed so the spread measured is training variance, not split variance,
and stays comparable to V12's fixed split.

| Config | What it tests |
|---|---|
| `v12` — base 32, 1x2x4, alpha 0.8 | the reference, now with an error bar |
| `winner` — base 48, 1x2x4x8, alpha 0.888, lr 8.2e-4 | is +4.16 dB real? |
| `winner_aug` — winner + D4 augmentation | does augmentation help? |

Then the best seed of each goes through the **all-5-holdout moment-map protocol**, because the
beam A/B already demonstrated that a pixel-metric win can coexist with a *worse* scientific
deliverable (M0 fell from +69.8% to +59.8% while PSNR rose).

**Cost.** Roughly 2.5–3 h GPU for the 9 runs on T4x2, plus ~10 min per moment-map evaluation.
Requires GPU. Run `07-classical-baselines.ipynb` in a separate CPU-only session in parallel —
it needs no GPU quota.

## 0. Bootstrap

In [ ]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'midterm-prep'
if ON_KAGGLE:
    REPO='/kaggle/working/EXXA'; PKG=os.path.join(REPO,'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    'pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks'));  sys.path.insert(0, PKG)
    hits = glob.glob('/kaggle/input/**/*_dirty.fits', recursive=True)
    DATA_DIR = os.path.dirname(os.path.dirname(hits[0])) if hits else None
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'): os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../data/Line Emission Data'
print('cwd:', os.getcwd(), '| DATA_DIR:', DATA_DIR)

## 0b. Pull latest code (re-run anytime — no kernel restart)

In [ ]:
import os, sys, subprocess
ON_KAGGLE = os.path.exists('/kaggle'); REPO = '/kaggle/working/EXXA'; BRANCH = 'midterm-prep'
if ON_KAGGLE and os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    print(subprocess.run(['git','-C',REPO,'log','--oneline','-1'],
                         capture_output=True, text=True).stdout.strip())
for _m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]
print('src.* cleared — re-run the imports cell below.')

## 1. Imports, device, config

In [ ]:
import csv, math, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from src.data.cube_split import split_cubes
from src.data.fits_cube_dataset import FITSChannelDataset, continuum_of
from src.models.unet import UNet
from src.training.sweep import repeat_config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU = torch.cuda.device_count()
SEED  = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('device:', device, '| GPUs:', N_GPU,
      '->', [torch.cuda.get_device_name(i) for i in range(N_GPU)] if N_GPU else 'cpu')

TARGET_SIZE, N_SAMPLES = 256, 150
SUBTRACT_CONTINUUM, CONTINUUM_N = True, 5
N_SEEDS = 3                     # floor for a std to mean anything; the std is itself noisy at 3
NW = 2 if ON_KAGGLE else 0
OUT_DIR, CKPT_DIR = '../results', '../results/checkpoints'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)

V12 = {'psnr': 32.95, 'ssim': 0.9857, 'mse': 0.000681,
       'M0': (69.8, 15.2), 'M1': (17.5, 7.8), 'M2': (20.1, 14.3)}
print(f'{N_SEEDS} seeds x 3 configs = {3*N_SEEDS} training runs')

## 2. Cube split — held FIXED across every seed

In [ ]:
train_cubes, val_cubes, holdout_cubes = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                     val_fraction=0.2, seed=SEED)
print('\nsplit is seeded separately and never varies per training seed, so the spread '
      'measured below is training variance on a fixed split.')

## 3. Datasets — one plain pair, one augmented train set

`val_ds` is shared by every run and has augmentation **off**: an augmented validation set makes
the early-stopping metric non-deterministic and the runs incomparable.

In [ ]:
common = dict(n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
              subtract_continuum=SUBTRACT_CONTINUUM, continuum_n=CONTINUUM_N)
train_ds     = FITSChannelDataset(train_cubes, **common)
train_ds_aug = FITSChannelDataset(train_cubes, augment=True, **common)
val_ds       = FITSChannelDataset(val_cubes,   **common)
print('train:', len(train_ds), '| train(aug):', len(train_ds_aug), '| val:', len(val_ds))

# sanity: augmentation must vary the train item and leave val deterministic
torch.manual_seed(0)
n_orient = len({train_ds_aug[0][0].numpy().tobytes() for _ in range(40)})
val_fixed = len({val_ds[0][0].numpy().tobytes() for _ in range(5)}) == 1
print(f'augmented train item takes {n_orient} distinct orientations | val deterministic: {val_fixed}')
assert n_orient > 1 and val_fixed, 'augmentation wiring is wrong -- stop and fix before training'

## 4. The three configurations

`v12` reproduces the reference architecture and loss, but under this notebook's early-stopping
schedule rather than V12's fixed 30 epochs — so it is a *re-measurement* of the reference under
a matched protocol, not a claim to reproduce V12's exact checkpoint. That matters: comparing the
winner against V12's published number alone would confound architecture with schedule.

In [ ]:
CONFIGS = {
    'v12':        dict(base_channels=32, channel_multipliers=(1, 2, 4),
                       lr=1e-3, alpha=0.8, sched_patience=5, use_beam=False, batch_size=32),
    'winner':     dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
                       lr=0.0008196504330730313, alpha=0.8877681051398497,
                       sched_patience=8, use_beam=False, batch_size=16),
}
CONFIGS['winner_aug'] = dict(CONFIGS['winner'])          # same config, augmented train set
DATASETS = {'v12': train_ds, 'winner': train_ds, 'winner_aug': train_ds_aug}

for name, cfg in CONFIGS.items():
    print(f'{name:<12} {cfg}')

## 5. Train — 3 configs x 3 seeds

Rows append to CSV as they finish, so a session timeout loses at most one run. Each seed's best
checkpoint is kept, so the moment-map protocol can run on a chosen seed rather than whatever
model happened to be last in memory.

In [ ]:
REPEAT_CSV = os.path.join(OUT_DIR, 'seed_repeats.csv')
results = {}
t_all = time.time()

for name, cfg in CONFIGS.items():
    t0 = time.time()
    results[name] = repeat_config(
        DATASETS[name], val_ds, device,
        n_seeds=N_SEEDS, base_seed=SEED, out_csv=REPEAT_CSV, ckpt_dir=CKPT_DIR,
        tag=name, num_workers=NW, min_epochs=20, max_epochs=60, patience=5,
        verbose=True, **cfg)
    print(f'--- {name} finished in {(time.time()-t0)/60:.1f} min ---\n', flush=True)

print(f'all {3*N_SEEDS} runs in {(time.time()-t_all)/60:.1f} min -> {REPEAT_CSV}')

## 6. Channel-level comparison, with error bars

In [ ]:
print('=' * 88)
print('{:<14} {:>22} {:>20} {:>22}'.format('config', 'PSNR (dB)', 'SSIM', 'MSE'))
print('-' * 88)
for name in CONFIGS:
    r = results[name]
    print('{:<14} {:>13.4f} +/- {:<6.4f} {:>11.4f} +/- {:<6.4f} {:>12.6f} +/- {:<8.6f}'.format(
        name, r['psnr']['mean'], r['psnr']['std'],
        r['ssim']['mean'], r['ssim']['std'], r['mse']['mean'], r['mse']['std']))
print('{:<14} {:>13.4f} {:>7} {:>11.4f} {:>7} {:>12.6f}'.format(
    'V12 (published)', V12['psnr'], '(n=1)', V12['ssim'], '(n=1)', V12['mse']))
print('=' * 88)

def verdict(a, b, name_a, name_b):
    """Compare two repeated configs, refusing to call a gap real if the spreads overlap."""
    ma, sa = a['psnr']['mean'], a['psnr']['std']
    mb, sb = b['psnr']['mean'], b['psnr']['std']
    gap = ma - mb
    pooled = float(np.sqrt(sa ** 2 + sb ** 2)) or 1e-12
    print(f'\n{name_a} - {name_b}: {gap:+.3f} dB  (spreads {sa:.3f} / {sb:.3f}, '
          f'combined {pooled:.3f})')
    if abs(gap) > 2 * pooled:
        print(f'  -> gap exceeds 2x the combined spread: treat as REAL')
    elif abs(gap) > pooled:
        print(f'  -> gap is 1-2x the combined spread: SUGGESTIVE, not established at n={N_SEEDS}')
    else:
        print(f'  -> gap is within the combined spread: INDISTINGUISHABLE from seed noise')
    return gap, pooled

verdict(results['winner'], results['v12'], 'winner', 'v12')
verdict(results['winner_aug'], results['winner'], 'winner_aug', 'winner')

print('\nNOTE: with n=3 the std is itself a noisy estimate. These thresholds are a guard '
      'against over-claiming, not a significance test.')

## 7. Loss curves — all seeds, all configs

In [ ]:
import pandas as pd
df = pd.read_csv(REPEAT_CSV)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, metric in zip(axes, ['psnr', 'ssim', 'mse']):
    for i, name in enumerate(CONFIGS):
        sub = df[df.tag == name]
        ax.scatter([i] * len(sub), sub[metric], s=60, alpha=0.8, zorder=5,
                   label=name if metric == 'psnr' else None)
        ax.errorbar(i, sub[metric].mean(), yerr=sub[metric].std(ddof=1),
                    fmt='_', ms=28, capsize=8, color='#333', zorder=4)
    if metric == 'psnr':
        ax.axhline(V12['psnr'], ls='--', color='#E8715A', lw=1.2, label='V12 published (n=1)')
        ax.legend(fontsize=8)
    ax.set_xticks(range(len(CONFIGS))); ax.set_xticklabels(list(CONFIGS), rotation=20, ha='right')
    ax.set_title(metric.upper()); ax.grid(axis='y', alpha=0.3)
fig.suptitle(f'Per-seed spread across {N_SEEDS} seeds (dots = seeds, bars = mean +/- std)',
             fontweight='bold')
plt.tight_layout()
p = os.path.join(OUT_DIR, 'seed_spread.png'); plt.savefig(p, dpi=140); plt.show()
print('saved ->', p)

## 8. Moment-map protocol on the best seed of each config

The scientific deliverable. Runs the same all-5-holdout evaluation V12 was judged on. The beam
A/B is the cautionary precedent: PSNR rose while M0 fell from +69.8% to +59.8% with doubled
variance, so a config is not promoted on pixel metrics alone.

In [ ]:
import bettermoments as bm
from astropy.io import fits
from src.evaluation.moment_maps import generate_moment_maps
from src.evaluation.classical import summarise_improvements

BS = 32
moments = ['M0', 'M1', 'M2']

def mdiff(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return float(np.nanmean(np.abs(a[m] - b[m])))

def load_net(ckpt_path):
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    net = UNet(in_channels=1, out_channels=1, base_channels=ck['base_channels'],
               channel_multipliers=ck['channel_multipliers'], time_emb_dim=128,
               num_res_blocks=2, groups=math.gcd(8, ck['base_channels']),
               beam_dim=ck.get('beam_dim', 0)).to(device)
    net.load_state_dict(ck['model_state_dict']); net.eval()
    return net, ck

def denoise_cube_unet(ho, net):
    with fits.open(ho['dirty'], memmap=False) as h:
        raw = np.ascontiguousarray(h[0].data).astype(np.float32)
    csub = raw - continuum_of(raw, CONTINUUM_N)[None]
    C, H, W = csub.shape
    lo = csub.reshape(C, -1).min(axis=1); hi = csub.reshape(C, -1).max(axis=1)
    rng_ = hi - lo; nz = rng_ > 0
    norm = np.zeros_like(csub)
    norm[nz] = (csub[nz] - lo[nz, None, None]) / rng_[nz, None, None]
    out = np.empty_like(csub)
    with torch.no_grad():
        for s in range(0, C, BS):
            t = torch.from_numpy(norm[s:s+BS])[:, None].to(device)
            t256 = F.interpolate(t, (TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
            tz = torch.zeros(t256.size(0), dtype=torch.long, device=device)
            pred = net(t256, tz)
            back = F.interpolate(pred, (H, W), mode='bilinear', align_corners=False)[:, 0].cpu().numpy()
            for k in range(back.shape[0]):
                ch = s + k
                out[ch] = back[k] * rng_[ch] + lo[ch] if rng_[ch] > 0 else np.full((H, W), lo[ch], np.float32)
    return out, csub

# cache clean/dirty moment maps once; reused by all three configs
cache = {}
for ho in holdout_cubes:
    with fits.open(ho['clean'], memmap=False) as h:
        craw = np.ascontiguousarray(h[0].data).astype(np.float32)
    ccsub = craw - continuum_of(craw, CONTINUUM_N)[None]
    with fits.open(ho['dirty'], memmap=False) as h:
        draw = np.ascontiguousarray(h[0].data).astype(np.float32)
    dcsub = draw - continuum_of(draw, CONTINUUM_N)[None]
    _, velax = bm.load_cube(ho['dirty'])
    cache[ho['folder']] = {'velax': velax,
                           'clean': generate_moment_maps(None, data_velax=(ccsub, velax)),
                           'dirty': generate_moment_maps(None, data_velax=(dcsub, velax))}
print('cached clean/dirty moment maps for', len(cache), 'cubes')

moment_summaries, moment_rows = {}, {}
for name in CONFIGS:
    best_seed = int(max(results[name]['rows'], key=lambda r: r['psnr'])['seed'])
    ckpt = os.path.join(CKPT_DIR, f'{name}_seed{best_seed}.pth')
    net, ck = load_net(ckpt)
    print(f'\n=== {name} (best seed {best_seed}, epoch {ck.get("epoch")}) ===', flush=True)
    rows = []
    for ho in holdout_cubes:
        den, _ = denoise_cube_unet(ho, net)
        e = cache[ho['folder']]
        no = generate_moment_maps(None, data_velax=(den, e['velax']))
        row = {'cube': ho['folder'], 'seed': best_seed}
        for nm, cl, di, n_ in zip(moments, e['clean'], e['dirty'], no):
            dd, nn = mdiff(cl, di), mdiff(cl, n_)
            row['imp_' + nm] = round(100.0 * (1 - nn / dd), 2) if dd > 0 else float('nan')
        rows.append(row)
        print('  {:<24} M0 {:>7.1f}%  M1 {:>7.1f}%  M2 {:>7.1f}%'.format(
            ho['folder'], row['imp_M0'], row['imp_M1'], row['imp_M2']))
    moment_rows[name] = rows
    moment_summaries[name] = summarise_improvements(rows)
    s = moment_summaries[name]
    print('  -> ' + '  '.join(f'{m} {s[m]["mean"]:+.1f}%+/-{s[m]["std"]:.1f}' for m in moments))
    del net
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 9. Final table — the midterm's decisive comparison

In [ ]:
print('=' * 96)
print('{:<20} {:>14} {:>18} {:>18} {:>18}'.format('config', 'PSNR (dB)', 'M0 (%)', 'M1 (%)', 'M2 (%)'))
print('-' * 96)
for name in CONFIGS:
    s, r = moment_summaries[name], results[name]
    print('{:<20} {:>8.2f}+/-{:<4.2f} {:>10.1f} +/-{:<5.1f} {:>10.1f} +/-{:<5.1f} {:>10.1f} +/-{:<5.1f}'.format(
        name, r['psnr']['mean'], r['psnr']['std'],
        s['M0']['mean'], s['M0']['std'], s['M1']['mean'], s['M1']['std'],
        s['M2']['mean'], s['M2']['std']))
print('{:<20} {:>8.2f}{:>6} {:>10.1f} +/-{:<5.1f} {:>10.1f} +/-{:<5.1f} {:>10.1f} +/-{:<5.1f}'.format(
    'V12 (published)', V12['psnr'], '(n=1)',
    V12['M0'][0], V12['M0'][1], V12['M1'][0], V12['M1'][1], V12['M2'][0], V12['M2'][1]))
print('{:<20} {:>8} {:>6} {:>10.1f} +/-{:<5.1f} {:>10.1f} +/-{:<5.1f} {:>10.1f} +/-{:<5.1f}'.format(
    'U-Net + beam', '34.94', '(n=1)', 59.8, 33.0, 19.2, 8.4, 23.2, 20.2))
print('=' * 96)

print('\nPromotion check — a config is only promoted if it wins on the SCIENTIFIC metric:')
ref = moment_summaries['v12']
for name in ('winner', 'winner_aug'):
    s = moment_summaries[name]
    print(f'\n  {name} vs v12 (this notebook, matched schedule):')
    for m in moments:
        gap = s[m]['mean'] - ref[m]['mean']
        pooled = float(np.sqrt(s[m]['std'] ** 2 + ref[m]['std'] ** 2)) or 1e-12
        flag = 'REAL' if abs(gap) > 2 * pooled else ('suggestive' if abs(gap) > pooled else 'within noise')
        print(f'    {m}: {gap:+6.1f} pp  (cube-to-cube spread {pooled:.1f} pp) -> {flag}')

csv_path = os.path.join(OUT_DIR, 'seed_moment_summary.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['config', 'seed', 'cube', 'imp_M0', 'imp_M1', 'imp_M2'])
    for name, rows in moment_rows.items():
        for r in rows:
            w.writerow([name, r['seed'], r['cube'], r['imp_M0'], r['imp_M1'], r['imp_M2']])
        s = moment_summaries[name]
        w.writerow([name, '', 'MEAN'] + [round(s[m]['mean'], 2) for m in moments])
        w.writerow([name, '', 'STD'] + [round(s[m]['std'], 2) for m in moments])
print('\nsaved ->', csv_path)

## 10. Artifact diagnostics — did augmentation reduce invented structure?

In [ ]:
from src.evaluation.artifacts import channel_artifacts, summarise

# The hallucination artifact was attributed to small data, which is exactly what
# augmentation is supposed to mitigate. This measures it rather than assuming it.
art = {}
for name in CONFIGS:
    best_seed = int(max(results[name]['rows'], key=lambda r: r['psnr'])['seed'])
    net, _ = load_net(os.path.join(CKPT_DIR, f'{name}_seed{best_seed}.pth'))
    rows = []
    with torch.no_grad():
        for i in range(len(val_ds)):
            d, c = val_ds[i]
            pred = net(d[None].to(device), torch.zeros(1, dtype=torch.long, device=device))
            cl = c[0].numpy()
            if cl.max() <= 0:
                continue
            rows.append(channel_artifacts(cl, d[0].numpy(), pred[0, 0].cpu().numpy()))
    art[name] = summarise(rows)
    a = art[name]
    print(f'{name:<12} overshoot {a["overshoot_mean"]:.3f} | floor leak {a["floor_leak_mean"]:+.5f} '
          f'| channels with invented blob {a["frac_channels_with_blob"]:.1%} '
          f'| blobs/channel {a["blobs_per_channel"]:.3f}')
    if 'low_snr_blobs_per_channel' in a:
        print(f'{"":12} low-SNR blobs/ch {a["low_snr_blobs_per_channel"]:.3f} vs '
              f'high-SNR {a["high_snr_blobs_per_channel"]:.3f} (split at SNR {a["snr_median"]:.1f})')
    del net
    if torch.cuda.is_available(): torch.cuda.empty_cache()

base = art['winner']['frac_channels_with_blob']
aug  = art['winner_aug']['frac_channels_with_blob']
print(f'\naugmentation effect on invented structure: {base:.1%} -> {aug:.1%} of channels '
      f'({aug - base:+.1%} pp)')

art_csv = os.path.join(OUT_DIR, 'artifact_diagnostics_seeds.csv')
keys = sorted({k for a in art.values() for k in a})
with open(art_csv, 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['config'] + keys)
    for name, a in art.items():
        w.writerow([name] + [a.get(k, '') for k in keys])
print('saved ->', art_csv)

## 11. Persist artifacts to /kaggle/working

In [ ]:
import shutil
if ON_KAGGLE:
    wanted = [REPEAT_CSV, csv_path, art_csv,
              os.path.join(OUT_DIR, 'seed_spread.png')]
    for name in CONFIGS:
        best_seed = int(max(results[name]['rows'], key=lambda r: r['psnr'])['seed'])
        wanted.append(os.path.join(CKPT_DIR, f'{name}_seed{best_seed}.pth'))
    for p in wanted:
        if os.path.exists(p):
            shutil.copy2(p, '/kaggle/working/' + os.path.basename(p))
            print('persisted ->', os.path.basename(p),
                  f'({os.path.getsize(p)/1e6:.1f} MB)')
    print('\ndownload the CSVs from the kernel Output tab and commit them to results/')
else:
    print('not on Kaggle — nothing to persist')